# Casino Revenue Reinforcement Learning Optimization

## Module 2 Data Science Capstone

This project uses reinforcement learning to explore decision-making strategies for optimizing casino game revenue. The RL agent will interact with a simulated environment based on the cleaned casino revenue dataset and learn which actions produce higher rewards.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/casino_game_revenue_revised_with_store_names.csv')

In [3]:
df.shape

(2000, 16)

In [4]:
df.head()

,machine_id,location_id,store_name,month,game_type,location_type,distance_from_hq_miles,days_active,avg_bet,payout_rate,total_in,total_out,total_net,number_of_services,machine_swapouts,monthly_revenue
0,M0017,L040,Eagle Ridge Gaming,2025-06-01,Classic Reels,Gaming Hall,13.9,27,1.49,0.897,1877.52,1689.65,187.87,2,1,-400.18
1,M0140,L003,Lone Star Game Center,2026-04-01,A-Liner,Convenience Store,57.5,30,1.74,0.948,1200.00,1145.11,54.89,1,0,362.46
2,M0118,L018,Copper Creek Gaming,2025-01-01,Life of Luxury,Restaurant,18.0,21,2.24,0.915,3833.53,3388.67,444.86,1,0,-206.00
3,M0079,L021,Cedar Grove Gaming,2025-09-01,Classic Reels,Gaming Hall,50.0,26,1.20,0.858,2241.71,1876.70,365.01,0,0,87.87
4,M0078,L036,Greenfield Game Center,2026-03-01,Pot of Gold,Gaming Hall,32.5,23,2.45,0.910,7874.12,6880.70,993.42,0,0,951.83


In [5]:
df.columns

Index(['machine_id', 'location_id', 'store_name', 'month', 'game_type',
       'location_type', 'distance_from_hq_miles', 'days_active', 'avg_bet',
       'payout_rate', 'total_in', 'total_out', 'total_net',
       'number_of_services', 'machine_swapouts', 'monthly_revenue'],
      dtype='str')

In [7]:
df.describe()

,distance_from_hq_miles,days_active,avg_bet,payout_rate,total_in,total_out,total_net,number_of_services,machine_swapouts,monthly_revenue
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,56.872450,25.524000,1.648400,0.910108,2463.218345,2243.627275,219.591070,0.758000,0.138500,46.593690
std,37.902925,3.462574,0.542625,0.024609,1481.670733,1354.796362,151.865676,0.907109,0.362469,593.981576
min,2.000000,20.000000,0.250000,0.824000,1200.000000,1000.930000,20.810000,0.000000,0.000000,-900.000000
25%,28.300000,22.000000,1.290000,0.893000,1341.440000,1226.165000,120.127500,0.000000,0.000000,-398.417500
50%,48.700000,26.000000,1.640000,0.910000,2040.800000,1841.435000,173.840000,1.000000,0.000000,40.655000
75%,77.025000,29.000000,2.000000,0.927000,2994.225000,2717.267500,270.295000,1.000000,0.000000,461.227500
max,180.000000,31.000000,3.360000,0.970000,13338.130000,12075.380000,1487.660000,6.000000,2.000000,2377.520000


## Reinforcement Learning Design

The reinforcement learning environment will model casino revenue optimization as a sequence of decisions.

- **State:** Current machine and operational conditions.
- **Action:** Operational decision selected by the agent.
- **Reward:** Revenue-based feedback used to evaluate the action.
- **Goal:** Learn a policy that maximizes cumulative revenue over time.

### State

The agent observes:

- `avg_bet`
- `payout_rate`
- `days_active`
- `number_of_services`
- `machine_swapouts`
- `monthly_revenue`

These features describe the current operating condition and performance of a casino gaming machine.

### Actions

The agent can choose one of three operational actions:

- **0 — Maintain:** Keep the current machine settings.
- **1 — Increase Bet:** Slightly increase the average bet.
- **2 — Adjust Payout:** Slightly adjust the payout rate.

The agent will learn which action produces the best long-term reward under different operating conditions.

### Reward

The reward represents the financial outcome of the agent's decision.

- Higher revenue produces a positive reward.
- Lower revenue produces a smaller reward.
- The agent's objective is to maximize cumulative reward over time.

`monthly_revenue` will be used as the primary signal for evaluating performance.

In [10]:
state_features = [
    'avg_bet',
    'payout_rate',
    'days_active',
    'number_of_services',
    'machine_swapouts',
    'monthly_revenue'
]

In [11]:
df[state_features].head()

,avg_bet,payout_rate,days_active,number_of_services,machine_swapouts,monthly_revenue
0,1.49,0.897,27,2,1,-400.18
1,1.74,0.948,30,1,0,362.46
2,2.24,0.915,21,1,0,-206.00
3,1.20,0.858,26,0,0,87.87
4,2.45,0.910,23,0,0,951.83


In [12]:
actions = {
    0: 'Maintain',
    1: 'Increase Bet',
    2: 'Adjust Payout'
}

In [13]:
actions

{0: 'Maintain', 1: 'Increase Bet', 2: 'Adjust Payout'}

In [14]:
state = df[state_features].iloc[0]

In [15]:
state


avg_bet                 1.490
payout_rate             0.897
days_active            27.000
number_of_services      2.000
machine_swapouts        1.000
monthly_revenue      -400.180
Name: 0, dtype: float64

In [16]:
reward = state['monthly_revenue']

In [17]:
reward

np.float64(-400.18)

In [18]:
action = 1

In [19]:
next_state = state.copy()

In [20]:
next_state['avg_bet'] *= 1.10

In [21]:
next_state[['avg_bet']]

avg_bet    1.639
Name: 0, dtype: float64

In [22]:
state['avg_bet']

np.float64(1.49)

In [23]:
new_reward = reward * 1.05

In [24]:
new_reward

np.float64(-420.189)

In [25]:
new_reward = reward + abs(reward) * 0.05


In [26]:
new_reward

np.float64(-380.171)

In [27]:
reward_change = new_reward - reward

In [28]:
reward_change

np.float64(20.009000000000015)

## RL Environment Logic

The next step converts the manually tested state-action-reward process into reusable environment logic. This allows the agent to apply different actions to different states consistently.

In [30]:
def apply_action(state, action):
    next_state = state.copy()

    if action == 1:
        next_state['avg_bet'] *= 1.10

    elif action == 2:
        next_state['payout_rate'] *= 0.99

    return next_state

In [31]:
test_maintain = apply_action(state, 0)

In [32]:
test_maintain.equals(state)

True

In [33]:
test_increase = apply_action(state, 1)

In [34]:
test_increase['avg_bet']

np.float64(1.639)

In [35]:
test_payout = apply_action(state, 2)

In [36]:
test_payout['payout_rate']

np.float64(0.88803)

In [37]:
print("Maintain:", test_maintain['avg_bet'], test_maintain['payout_rate'])
print("Increase Bet:", test_increase['avg_bet'], test_increase['payout_rate'])
print("Adjust Payout:", test_payout['avg_bet'], test_payout['payout_rate'])

Maintain: 1.49 0.897
Increase Bet: 1.639 0.897
Adjust Payout: 1.49 0.88803


## Reward Function

The reward function measures the change in simulated revenue after an action.

A positive reward indicates an improvement in the simulated outcome, while a negative reward indicates a deterioration.

The reward is based on the difference between the new simulated revenue and the current revenue.

In [38]:
def calculate_reward(old_revenue, new_revenue):
    return new_revenue - old_revenue

In [39]:
calculate_reward(-400.18, -380.171)

20.009000000000015

In [40]:
calculate_reward(-400.18, -420.189)


-20.009000000000015

In [41]:
print(calculate_reward(-400.18,-380.171))
print(calculate_reward(-400.18,-420.189))

20.009000000000015
-20.009000000000015
